In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/esm2-variant-benchmark'

import re
import pandas as pd
from tqdm import tqdm
print('Ready.')


Mounted at /content/drive
Ready.


In [2]:
# fair-esm: Meta's official library for ESM protein language models.
# Confirm a GPU is attached before proceeding — this step is very slow on CPU.
import torch
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then rerun.')

!pip install -q fair-esm

GPU available: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 4.5 MB/s eta 0:00:00


In [3]:
# esm2_t33_650M_UR50D: the 650M-parameter ESM-2 model, a good balance of
# accuracy and speed for Colab's free-tier GPU. Downloads ~2.5GB on first run.
import esm

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
batch_converter = alphabet.get_batch_converter()
print(f'Model loaded on {device}.')

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Model loaded on cuda.


In [4]:
# Load the canonical sequences (keyed by accession) and the variant table
# with conservation scores already attached from WP2.
!pip install -q biopython
from Bio import SeqIO

sequences_fasta = f'{PROJECT_DIR}/data/raw/sequences.fasta'
accession_to_seq = {}
for record in SeqIO.parse(sequences_fasta, 'fasta'):
    accession = record.id.split('|')[1] if '|' in record.id else record.id
    accession_to_seq[accession] = str(record.seq)

merged_path = f'{PROJECT_DIR}/data/processed/variants_with_conservation.tsv'
variants_df = pd.read_csv(merged_path, sep='\t')
print(f'{len(variants_df)} variants across {variants_df["uniprot_accession"].nunique()} proteins loaded.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.9 MB/s eta 0:00:00
6238 variants across 20 proteins loaded.


In [5]:
# protein_change looks like "Arg175His" -- split into wild-type residue,
# position, and mutant residue, then convert 3-letter to 1-letter codes.
aa3to1 = {
    'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Gln':'Q','Glu':'E',
    'Gly':'G','His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F',
    'Pro':'P','Ser':'S','Thr':'T','Trp':'W','Tyr':'Y','Val':'V'
}
aa3 = '|'.join(aa3to1.keys())
change_re = re.compile(rf'({aa3})(\d+)({aa3})')

def parse_change(protein_change):
    match = change_re.match(str(protein_change))
    if not match:
        return None, None, None
    wt3, pos, mut3 = match.groups()
    return aa3to1[wt3], int(pos), aa3to1[mut3]

parsed = variants_df['protein_change'].apply(parse_change)
variants_df['wt_aa'] = parsed.apply(lambda x: x[0])
variants_df['mut_aa'] = parsed.apply(lambda x: x[2])

variants_df = variants_df.dropna(subset=['wt_aa', 'mut_aa'])
print(f'{len(variants_df)} variants successfully parsed for scoring.')

6238 variants successfully parsed for scoring.


In [6]:
# Run this in a new cell to confirm the model is really on GPU
print(next(model.parameters()).device)

cuda:0


In [13]:
# For each protein, batch together one masked copy of a WINDOW around each
# unique variant position (not the whole sequence), run through ESM-2, and
# read off log P(mutant) - log P(wild-type) at each masked position.
# Windowing lets us handle long proteins (BRCA2, ATM, etc.) that would
# otherwise get skipped or blow past GPU memory limits.
BATCH_SIZE = 8       # lower if you hit an out-of-memory error
WINDOW = 500         # residues kept on each side of the variant position
MAX_MODEL_LEN = 1022 # ESM-2's hard input limit; window*2+1 must stay under this

esm2_records = []

def get_windowed_sequence(sequence, pos):
    start = max(0, pos - WINDOW - 1)
    end = min(len(sequence), pos + WINDOW)
    new_pos = pos - start  # 1-indexed position of the variant within the window
    return sequence[start:end], new_pos

for accession, group in tqdm(variants_df.groupby('uniprot_accession')):
    print(f'Starting {accession} ({len(group)} variants)...', flush=True)
    sequence = accession_to_seq.get(accession)
    if sequence is None:
        print(f'  Skipping {accession}: no sequence found.')
        continue

    unique_positions = group[['position', 'wt_aa']].drop_duplicates()
    position_list = list(unique_positions.itertuples(index=False))

    position_to_score = {}

    for i in range(0, len(position_list), BATCH_SIZE):
        batch_positions = position_list[i:i + BATCH_SIZE]
        batch_data = []
        batch_new_pos = []

        for pos_row in batch_positions:
            pos = pos_row.position
            windowed_seq, new_pos = get_windowed_sequence(sequence, pos)
            masked_seq = windowed_seq[:new_pos - 1] + '<mask>' + windowed_seq[new_pos:]
            batch_data.append((f'{accession}_{pos}', masked_seq))
            batch_new_pos.append(new_pos)

        _, _, batch_tokens = batch_converter(batch_data)
        batch_tokens = batch_tokens.to(device)

        with torch.no_grad():
            logits = model(batch_tokens)['logits']
            log_probs = torch.log_softmax(logits, dim=-1)

        for j, pos_row in enumerate(batch_positions):
            pos = pos_row.position
            mask_idx = (batch_tokens[j] == alphabet.mask_idx).nonzero(as_tuple=True)[0]
            if len(mask_idx) == 0:
                continue
            mask_idx = mask_idx.item()
            position_to_score[pos] = log_probs[j, mask_idx]

    for _, row in group.iterrows():
        pos = row['position']
        if pos not in position_to_score:
            continue
        log_probs_at_pos = position_to_score[pos]
        wt_idx = alphabet.get_idx(row['wt_aa'])
        mut_idx = alphabet.get_idx(row['mut_aa'])
        score = (log_probs_at_pos[mut_idx] - log_probs_at_pos[wt_idx]).item()
        esm2_records.append({
            'uniprot_accession': accession,
            'position': pos,
            'wt_aa': row['wt_aa'],
            'mut_aa': row['mut_aa'],
            'esm2_score': score
        })

esm2_df = pd.DataFrame(esm2_records)
esm2_path = f'{PROJECT_DIR}/data/processed/esm2_scores.tsv'
esm2_df.to_csv(esm2_path, sep='\t', index=False)
print(f'{len(esm2_df)} variants scored with ESM-2.')
esm2_df.head()


  0%|          | 0/20 [00:00<?, ?it/s]

Starting P00439 (470 variants)...


  5%|▌         | 1/20 [00:54<17:17, 54.63s/it]

Starting P00533 (52 variants)...


 10%|█         | 2/20 [01:21<11:31, 38.39s/it]

...

 95%|█████████▌| 19/20 [28:42<01:36, 96.49s/it] 

Starting Q13315 (218 variants)...


100%|██████████| 20/20 [30:31<00:00, 91.59s/it] 

6238 variants scored with ESM-2.


,uniprot_accession,position,wt_aa,mut_aa,esm2_score
0,P00439,408,R,W,-7.097335
1,P00439,311,L,P,-10.732636
2,P00439,280,E,K,-7.886076
3,P00439,261,R,Q,-4.044849
4,P00439,87,S,R,-1.776208


In [14]:
final_df = variants_df.merge(
    esm2_df[['uniprot_accession', 'position', 'wt_aa', 'mut_aa', 'esm2_score']],
    on=['uniprot_accession', 'position', 'wt_aa', 'mut_aa'], how='left'
)

missing = final_df['esm2_score'].isna().sum()
print(f'{missing} of {len(final_df)} variants missing an ESM-2 score')

final_path = f'{PROJECT_DIR}/data/processed/variants_with_all_scores.tsv'
final_df.to_csv(final_path, sep='\t', index=False)
final_df.head()

0 of 6498 variants missing an ESM-2 score


,GeneSymbol,uniprot_accession,protein_change,label,ReviewStatus,PhenotypeList,position,conservation_score,wt_aa,mut_aa,esm2_score
0,PAH,P00439,Arg408Trp,1,reviewed by expert panel,Phenylketonuria|not provided|Inborn genetic di...,408,0.667587,R,W,-7.097335
1,PAH,P00439,Leu311Pro,1,reviewed by expert panel,Phenylketonuria|not provided,311,0.737689,L,P,-10.732636
2,PAH,P00439,Glu280Lys,1,"criteria provided, multiple submitters, no con...","Phenylketonuria|not provided|Polymicrogyria, p...",280,0.834147,E,K,-7.886076
3,PAH,P00439,Arg261Gln,1,reviewed by expert panel,Phenylketonuria|not provided|PAH-related disorder,261,0.779125,R,Q,-4.044849
4,PAH,P00439,Ser87Arg,1,reviewed by expert panel,Hyperphenylalaninemia|not provided|Phenylketon...,87,0.405575,S,R,-1.776208


In [15]:
# ESM-2 scores should be more negative (mutant less likely than wild-type)
# for pathogenic variants than benign ones -- same directional logic as
# the conservation score, just from a completely different method.
print(final_df.groupby('label')['esm2_score'].describe())

        count      mean       std        min        25%       50%       75%  \
label                                                                         
0      2113.0 -1.884469  3.028430 -14.436807  -3.210493 -0.804530  0.132945   
1      4385.0 -8.962179  3.639266 -17.513733 -11.706189 -9.851082 -6.746157   

            max  
label            
0      4.946730  
1      4.730687  
